# 03d — CP2 timezone-v2 production migration / refreeze audit

Mục tiêu: promote contract đã audit trong notebook 03 vào production `src/` mà **không nghiên cứu lại timezone từ đầu**.

Gate này so sánh production-v2 với independent reference được chép từ final notebook-03 path trên cùng frozen 5,821 CP1 stays:

- per-stay IANA timezone exact;
- local wall-clock timestamps exact;
- complete-link 200m per-stay `location_id` exact;
- emitted `(user,label,location_id)` exact;
- share/margin/date/evidence exact;
- HTTP stay-event endpoint parity với direct production.

Không có Beijing radius / China eligibility gate trong candidate v2.


In [ ]:
from pathlib import Path
import importlib.util, json, os, subprocess, sys

REPO_URL='https://github.com/tanh1c/geolife.git'
BRANCH='feat/cp2-timezone-v2-production-migration'
REPO_DIR=Path('/root/geolife')

if not (REPO_DIR/'.git').exists():
    subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REPO_URL,str(REPO_DIR)],check=True)
else:
    subprocess.run(['git','-C',str(REPO_DIR),'fetch','origin',BRANCH],check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'checkout',BRANCH],check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'reset','--hard',f'origin/{BRANCH}'],check=True)

subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[dev]'],cwd=REPO_DIR,check=True)
for p in (str(REPO_DIR),str(REPO_DIR/'src')):
    if p not in sys.path: sys.path.insert(0,p)
sha=subprocess.run(['git','-C',str(REPO_DIR),'rev-parse','HEAD'],capture_output=True,text=True,check=True).stdout.strip()
print('sha:',sha)


In [ ]:
import pandas as pd
from fastapi.testclient import TestClient

from geolife.model import HomeOfficeConfig, build_semantic_locations, infer_home_office, location_namespace
from geolife.api.app import app

spec=importlib.util.spec_from_file_location('cp2mig',REPO_DIR/'analysis'/'03d_cp2_timezone_v2_migration.py')
cp2mig=importlib.util.module_from_spec(spec); sys.modules[spec.name]=cp2mig; spec.loader.exec_module(cp2mig)

V=Path('/mnt/geolife-data')
if not V.exists(): raise FileNotFoundError('Attach GeoLife Modal Volume at /mnt/geolife-data')
C=V/'cache'/'03d_cp2_timezone_v2_migration'; C.mkdir(parents=True,exist_ok=True)

candidates=[
    V/'cache'/'03a_user_behavior_deep_dive'/'stays_baseline_v1.pkl',
    V/'artifacts'/'03a'/'stays_baseline_v1.pkl',
    V/'cache'/'cp2_home_office'/'stays_baseline_v1.pkl',
]
STAYS=next((p for p in candidates if p.exists()),None)
if STAYS is None:
    matches=sorted(V.glob('**/stays_baseline_v1.pkl'))
    if not matches: raise FileNotFoundError('stays_baseline_v1.pkl not found')
    STAYS=matches[0]
stays=pd.read_pickle(STAYS)
print('stay cache:',STAYS)
print('stays:',len(stays),'users:',stays['user_id'].nunique())
assert len(stays)==5821
assert stays['user_id'].nunique()==136
cfg=HomeOfficeConfig()
print('namespace:',location_namespace(cfg))
assert location_namespace(cfg)=='complete_link_200m_local_timezone_v2'


## A. Independent notebook-v2 reference parity

Reference implementation nằm trong `analysis/03d...` và không gọi production private helpers. Đây là migration comparator, không phải implementation thứ hai để dùng lâu dài.


In [ ]:
production_semantic,production_locations=build_semantic_locations(stays,config=cfg)
production_labels=infer_home_office(stays,config=cfg)

reference_labels,reference_semantic,reference_locations=cp2mig.reference_infer_home_office(stays)

summary,decision=cp2mig.compare_migration(
    stays=stays,
    production_labels=production_labels,
    production_semantic=production_semantic,
    production_locations=production_locations,
    reference_labels=reference_labels,
    reference_semantic=reference_semantic,
    reference_locations=reference_locations,
)
display(summary.T.rename(columns={0:'value'}))
display(cp2mig.decision_frame(decision))

print('\nTimezone distribution:')
display(production_semantic['timezone_id'].value_counts().rename('stays').to_frame().head(20))

print('\nProduction v2 emissions:')
display(production_labels['label'].value_counts().reindex(['HOME','OFFICE'],fill_value=0).rename('emitted').to_frame())

if decision.runner_status!='ready_to_refreeze_cp2_v2':
    raise RuntimeError(f'CP2 v2 reference parity blocked: {decision}')


## B. Full-release HTTP ↔ direct-model parity

Replay từng user qua `/v1/home-office/infer`. Không lưu raw coordinates/timestamps vào artifact; chỉ lưu aggregate mismatch counts.


In [ ]:
client=TestClient(app)
http_mismatch_users=[]
contract_mismatch_users=[]

for user_id,user_stays in stays.groupby('user_id',sort=True):
    direct=infer_home_office(user_stays,config=cfg)
    payload={
        'user_id':str(user_id),
        'stays':[
            {
                'arrival_time_utc':pd.Timestamp(row.arrival_time_utc).isoformat(),
                'departure_time_utc':pd.Timestamp(row.departure_time_utc).isoformat(),
                'latitude':float(row.latitude),
                'longitude':float(row.longitude),
            }
            for row in user_stays.itertuples(index=False)
        ],
    }
    response=client.post('/v1/home-office/infer',json=payload)
    if response.status_code!=200:
        http_mismatch_users.append(str(user_id)); continue
    body=response.json()
    if body.get('model_contract')!='cp2-v2':
        contract_mismatch_users.append(str(user_id))
    emitted={item['label']:item for item in body['results'] if item['status']=='emitted'}
    direct_map={row.label:row for row in direct.itertuples(index=False)}
    if set(emitted)!=set(direct_map):
        http_mismatch_users.append(str(user_id)); continue
    for label,row in direct_map.items():
        item=emitted[label]
        exact=(
            int(item['location_id'])==int(row.location_id)
            and abs(float(item['evidence_strength'])-float(row.evidence_strength))<=1e-12
            and abs(float(item['relevant_dwell_share'])-float(row.relevant_dwell_share))<=1e-12
            and abs(float(item['share_margin'])-float(row.share_margin))<=1e-12
            and int(item['relevant_dates'])==int(row.relevant_dates)
            and abs(float(item['relevant_dwell_h'])-float(row.relevant_dwell_h))<=1e-12
        )
        if not exact:
            http_mismatch_users.append(str(user_id)); break

api_summary=pd.DataFrame([{
    'users_replayed':int(stays['user_id'].nunique()),
    'http_mismatch_users':len(set(http_mismatch_users)),
    'model_contract_mismatch_users':len(set(contract_mismatch_users)),
    'http_direct_exact':not http_mismatch_users and not contract_mismatch_users,
}])
display(api_summary.T.rename(columns={0:'value'}))
if not bool(api_summary.iloc[0]['http_direct_exact']):
    raise RuntimeError('HTTP/direct CP2 v2 parity failed')


## C. Freeze private migration artifacts

Không ghi coordinates, raw timestamps hoặc user-level mismatches ra repo.


In [ ]:
final_summary=summary.assign(
    distinct_timezones=int(production_semantic['timezone_id'].nunique()),
    unique_emitted_users=int(production_labels['user_id'].nunique()),
    location_namespace=location_namespace(cfg),
    http_direct_exact=bool(api_summary.iloc[0]['http_direct_exact']),
    migration_sha=sha,
)
final_summary.to_csv(C/'migration_summary.csv',index=False)
cp2mig.decision_frame(decision).to_csv(C/'migration_decision.csv',index=False)
api_summary.to_csv(C/'api_parity_summary.csv',index=False)
production_labels.drop(columns=['latitude','longitude'],errors='ignore').to_pickle(C/'production_v2_labels_private.pkl')
production_locations.drop(columns=['latitude','longitude'],errors='ignore').to_pickle(C/'production_v2_locations_private.pkl')

manifest={
    'model_contract':'cp2-v2',
    'location_namespace':location_namespace(cfg),
    'migration_sha':sha,
    'cp1_stays':int(len(stays)),
    'cp1_users':int(stays['user_id'].nunique()),
    'semantic_stays':int(len(production_semantic)),
    'semantic_users':int(production_semantic['user_id'].nunique()),
    'distinct_timezones':int(production_semantic['timezone_id'].nunique()),
    'locations':int(len(production_locations)),
    'recurring_locations':int((production_locations['stay_count']>=2).sum()),
    'recurring_users':int(production_locations.loc[production_locations['stay_count']>=2,'user_id'].nunique()),
    'home_emitted':int((production_labels['label']=='HOME').sum()),
    'office_emitted':int((production_labels['label']=='OFFICE').sum()),
    'unique_emitted_users':int(production_labels['user_id'].nunique()),
    'reference_parity':decision.runner_status=='ready_to_refreeze_cp2_v2',
    'http_direct_parity':bool(api_summary.iloc[0]['http_direct_exact']),
    'runner_status':'ready_to_refreeze_cp2_v2',
}
(C/'migration_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
print('saved:',C)
print(json.dumps(manifest,ensure_ascii=False,indent=2))


## Exit criterion

Chỉ refreeze production CP2 v2 nếu:

```text
reference parity = exact
HTTP/direct parity = exact
runner_status = ready_to_refreeze_cp2_v2
```

Sau khi pass, downstream artifacts dựa trên legacy `build_semantic_locations()` phải rebuild trên namespace mới; CP1 và source-level BCL acquisition 07f không cần chạy lại.
